# Sprint 2

## python
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install --upgrade pip

# Этап 1. Аудит и подготовка корпуса

## pip
```
pip install ipykernel 
pip install tqdm ipywidgets
pip install ftfy bs4 trafilatura
python -m pip install urllib3[socks]
```

## dataset
wget https://code.s3.yandex.net/ai-engineer/Итоговый_проект_спринт2/filmoskop_kb.json -O filmoskop_kb.json

In [145]:
'''
1.0 Load primary docs
'''
import json

with open("filmoskop_kb.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]        # 181 раздел базы знаний
queries = data["queries"]  # 66 тестовых запросов с разметкой

print("Разделов:", len(docs), "| Запросов:", len(queries))

Разделов: 181 | Запросов: 66


In [146]:
'''
1.1 count junk_ratio - pre
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.05  | Junk length:  7079  | Text length:  145549


In [147]:
'''
1.2 filter HTML
'''
from bs4 import BeautifulSoup  
import trafilatura

def normalize_html(raw):
  soup = BeautifulSoup(raw, "html.parser")
  for tag in soup(["script", "style", "nav", "footer"]):
      tag.decompose()
  for node in soup.select(".cookie"):
      node.decompose()
  return soup.get_text(" ")

for t in docs:
  if t['format'] == 'html':
    txt = trafilatura.extract(t['raw'], favor_precision=False, include_comments=False) 
    if not txt:
    #  print(t['doc_id'])
     txt = normalize_html(t['raw'])
    t['raw'] = txt

# filtered = [i for i in docs if i['raw']]

# write to file
with open("tmp/docs_html.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

In [148]:
'''
1.3 filter Markdown
'''
import re

for t in docs:
  if t['format'] == 'markdown':
    t['raw'] = re.sub(r"^#.*\n\n", "", t['raw'], flags=re.MULTILINE)
    t['raw'] = re.sub(r"^#.*\n", "", t['raw'], flags=re.MULTILINE)
    t['raw'] = re.sub(r"^#{1,6}\s+", "", t['raw'], flags=re.MULTILINE)

# write to file
with open("tmp/docs_md.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

In [149]:
'''
1.4 encoding, normalize text
'''
import re
import unicodedata
import ftfy

def encode_text(text):
  # Если текст побит — восстанавливаем «кракозябры»
  return ftfy.fix_text(text)

def normalize_text(text):
  # Приводим юникод к единой форме NFC
  text = unicodedata.normalize("NFC", text)

  # Склеиваем слово, разорванное переносом строки
  text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

  # Неразрывные и нулевой ширины пробелы -> обычный пробел
  text = text.replace("\N{NO-BREAK SPACE}", " ")
  text = text.replace("\N{ZERO WIDTH SPACE}", " ")

  # Схлопываем подряд идущие пробелы и переносы в один пробел
  text = re.sub(r"\s+", " ", text)

  return text.strip()


for t in docs:
  if not t['raw']:
    print('>>>', t)
    continue
  if t['format'] != 'html':
    t['raw'] = encode_text(t['raw'])

In [150]:
'''
1.5 remove useless data and put period in the end of item
'''
MIN_LENGTH = 30

filtered = []
for i in docs:
  if len(i['raw']) < MIN_LENGTH \
  or 'Внимание: статья устарела' in i['raw'] \
  or 'выгружено автоматически' in i['raw'] \
  or ' — не публиковать' in i['raw'] \
  or 'Раздел временно недоступен' in i['raw'] \
  or 'Страница не найдена' in i['raw'] \
  or 'Раздел в разработке' in i['raw'] \
  or 'Не pедактировать вpучную' in i['raw'] \
  or 'ORDER BY' in i['raw'] \
  or 'Черновик.' in i['raw'] \
  or 'черновик, не публиковать' in i['raw']:
    continue
  
  i['raw'] = i['raw'].replace("Статья была полезна? Да Нет", "").strip()
  # i['raw'] = i['raw'].replace("— Справка «Фильмоскопа» Главная → Справка →", "").strip()
  # i['raw'] = i['raw'].replace("Мы используем cookie, чтобы сервис работал лучше", "").strip()
  i['raw'] = i['raw'].replace("→", "-").strip()

  if not i['raw'].endswith("."):
    i['raw'] = f"{i['raw']}." 

  filtered.append(i)

for t in filtered:
  t['raw'] = normalize_text(t['raw'])

with open("tmp/docs_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=4)

In [160]:
'''
1.6 Deduplication
'''
import difflib
import itertools

# Recommended Default:
# Catches minor typos, plural forms, or single-word tweaks while keeping distinct sentences separate
threshold=0.85

relevant_docs = [i['relevant_docs'] for i in queries]
relevant_docs = sorted(list(set((list(itertools.chain.from_iterable(relevant_docs))))))

unique = [i['raw'] for i in filtered if i['doc_id'] in relevant_docs]
final = [i for i in filtered if i['doc_id'] in relevant_docs]
items = [i for i in filtered if i not in final]

duplicates = []
for item in items:
  is_duplicate = any(
    difflib.SequenceMatcher(None, item['raw'], existing).ratio() >= threshold
    for existing in unique
  )
  if is_duplicate:
    duplicates.append(item)
    print('>>> duplicate: ', item)
  else:
    unique.append(item['raw'])
    final.append(item)

final_sorted = sorted(final, key=lambda x: int(x["doc_id"]))

with open("1_corpus.json", "w", encoding="utf-8") as f:
    json.dump(final_sorted, f, ensure_ascii=False, indent=4)

>>> duplicate:  {'doc_id': 77, 'title': 'Как смотреть с телефона на большом экране', 'category': 'Устройства и качество', 'format': 'text', 'raw': 'Видео из приложения на телефоне можно отправить на большой экран. Через Chromecast: телефон и телевизор должны быть в одной Wi-Fi сети; нажмите значок трансляции в плеере и выберите устройство - воспроизведение продолжится на телевизоре, а телефон станет пультом. Через AirPlay: на iPhone и iPad нажмите значок AirPlay в плеере и выберите Apple TV или телевизор с поддержкой AirPlay 2. Остановить трансляцию можно тем же значком.'}
>>> duplicate:  {'doc_id': 86, 'title': 'Поддерживаемые устройства', 'category': 'Устройства и качество', 'format': 'markdown', 'raw': 'На каких устройствах работает "Фильмоскоп" Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4.0 и новее), телевизоры и приставки на Android TV 9 и новее. ТВ-приставки: Android TV, Apple TV (tvOS 15 и новее), приставки операторов-партнеров. Телефоны и планшеты: iOS 

In [161]:
'''
1.6.1 Lost relevant_docs
'''
import itertools

doc_ids = [i['doc_id'] for i in final]

relevant_docs = [i['relevant_docs'] for i in queries]
relevant_docs = list(itertools.chain.from_iterable(relevant_docs))

lost = [i for i in relevant_docs if i not in doc_ids]

print(lost)

[]


In [162]:
'''
1.6 Check junk_ratio
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)

  # print('>>> junk', junk)
  return len(junk) 


text_len = 0
junk_len = 0
for t in final:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), 
      ' | Junk length: ', junk_len, 
      ' | Text length: ', text_len,
      ' | Разделов: ', len(final),
      )

Junk ratio:  0.0  | Junk length:  42  | Text length:  110439  | Разделов:  139


In [163]:

'''
1.7 Corpus Statistics
'''
MIN_LENGTH = 30

texts = [item["raw"] for item in final]

total = 0
total_length = 0
count = 0
for item in texts:
  total += 1
  total_length += len(item)
  if len(item) < MIN_LENGTH:
    count += 1
    print(item)

shortest = min(texts, key=len)
longest = max(texts, key=len)

print(
  '>>> suspicious lengths: ', count, 
  ' | Mean length: ', round(total_length/total, 2), 
  ' | min: ', len(shortest), 
  ' | longest: ', len(longest),
  ' | total docs: ', len(texts),
)
# print('>>> longest:', longest)

>>> suspicious lengths:  0  | Mean length:  794.53  | min:  70  | longest:  1830  | total docs:  139


# Этап 1. Аудит и подготовка корпуса - Report

## Исходный датасет
Разделов: 181 | Запросов: 66 | Text length:  145549

Форматы:
- text
- html

Junk:
- Junk ratio:  0.05  
- Junk length:  7079  


## Transformations
- HTML -> TEXT = trafilatura + BeautifulSoup
- MD -> TEXT
- Encoding unification
- Text normalization
- Query normalization

- Deduplication
threshold=0.85
Recommended Default: Catches minor typos, plural forms, or single-word tweaks while keeping distinct sentences separate

- Suspicious lengths
MIN_LENGTH = 30 - manual check for useful texts

- Junk symbols 
JUNK_THRESHOLD = 0.15

- Manual audit


## Final result
- Suspicious lengths:  0  
- Mean length:  795
- min:  70  / longest:  1830
- total docs:  139
- Junk ratio:  0.0  | Junk length:  42
- No duplicates

## Examples
### text
- Initial:
{
  "doc_id": 43,
  "title": "Аудиодорожки и оригинальная озвучка",
  "category": "Функции и контент",
  "format": "text",
  "raw": "Ð£ Ð¼Ð½Ð¾Ð³Ð¸Ñ ÑÐ¸Ð»ÑÐ¼Ð¾Ð² Ð¸ ÑÐµÑÐ¸Ð°Ð»Ð¾Ð² ÐµÑÑÑ Ð½ÐµÑÐºÐ¾Ð»ÑÐºÐ¾ Ð°ÑÐ´Ð¸Ð¾Ð´Ð¾ÑÐ¾Ð¶ÐµÐº: ÑÑÑÑÐºÐ°Ñ Ð¾Ð·Ð²ÑÑÐºÐ°, Ð¾ÑÐ¸Ð³Ð¸Ð½Ð°Ð»ÑÐ½Ð°Ñ Ð´Ð¾ÑÐ¾Ð¶ÐºÐ°, Ð¸Ð½Ð¾Ð³Ð´Ð° â Ð´Ð¾ÑÐ¾Ð¶ÐºÐ¸ Ð½Ð° Ð´ÑÑÐ³Ð¸Ñ ÑÐ·ÑÐºÐ°Ñ.\n\nÐÐµÑÐµÐºÐ»ÑÑÐ¸ÑÑ Ð´Ð¾ÑÐ¾Ð¶ÐºÑ Ð¼Ð¾Ð¶Ð½Ð¾ Ð¿ÑÑÐ¼Ð¾ Ð² Ð¿Ð»ÐµÐµÑÐµ: Ð¾ÑÐºÑÐ¾Ð¹ÑÐµ Ð¼ÐµÐ½Ñ Â«ÐÑÐ´Ð¸Ð¾Â» Ð¸ Ð²ÑÐ±ÐµÑÐ¸ÑÐµ Ð½ÑÐ¶Ð½ÑÑ. ÐÑÐ±Ð¾Ñ Ð·Ð°Ð¿Ð¾Ð¼Ð¸Ð½Ð°ÐµÑÑÑ Ð´Ð»Ñ ÐºÐ¾Ð½ÐºÑÐµÑÐ½Ð¾Ð³Ð¾ ÑÐ¸Ð»ÑÐ¼Ð°. ÐÐ°Ð±Ð¾Ñ Ð´Ð¾ÑÐ¾Ð¶ÐµÐº Ð·Ð°Ð²Ð¸ÑÐ¸Ñ Ð¾Ñ Ð¿ÑÐ°Ð² Ð½Ð° ÐºÐ¾Ð½ÑÐµÐ½Ñ, Ð¿Ð¾ÑÑÐ¾Ð¼Ñ Ð¾ÑÐ¸Ð³Ð¸Ð½Ð°Ð»ÑÐ½Ð°Ñ Ð¾Ð·Ð²ÑÑÐºÐ° Ð´Ð¾ÑÑÑÐ¿Ð½Ð° Ð½Ðµ Ð²ÐµÐ·Ð´Ðµ: Ð½Ð°Ð»Ð¸ÑÐ¸Ðµ Ð´Ð¾ÑÐ¾Ð¶ÐµÐº ÑÐºÐ°Ð·Ð°Ð½Ð¾ Ð² ÐºÐ°ÑÑÐ¾ÑÐºÐµ ÑÐ¸Ð»ÑÐ¼Ð°. ÐÐ°ÑÑÑÐ¾Ð¹ÐºÐ° Â«ÐÑÐµÐ´Ð¿Ð¾ÑÐ¸ÑÐ°ÐµÐ¼ÑÐ¹ ÑÐ·ÑÐº Ð°ÑÐ´Ð¸Ð¾Â» Ð² Ð¿ÑÐ¾ÑÐ¸Ð»Ðµ Ð·Ð°Ð´Ð°ÑÑ Ð´Ð¾ÑÐ¾Ð¶ÐºÑ Ð¿Ð¾ ÑÐ¼Ð¾Ð»ÑÐ°Ð½Ð¸Ñ."
}
- Final:
{
  "doc_id": 43,
  "title": "Аудиодорожки и оригинальная озвучка",
  "category": "Функции и контент",
  "format": "text",
  "raw": "У многих фильмов и сериалов есть несколько аудиодорожек: русская озвучка, оригинальная дорожка, иногда — дорожки на других языках. Переключить дорожку можно прямо в плеере: откройте меню «Аудио» и выберите нужную. Выбор запоминается для конкретного фильма. Набор дорожек зависит от прав на контент, поэтому оригинальная озвучка доступна не везде: наличие дорожек указано в карточке фильма. Настройка «Предпочитаемый язык аудио» в профиле задаёт дорожку по умолчанию."
}
### html
- Initial:
{
  "doc_id": 57,
  "title": "Политика обработки персональных данных (выдержка)",
  "category": "Аккаунт и профили",
  "format": "html",
  "raw": "<div class=\"kb-article\"><h1>Политика обработки персональных данных (выдержка)</h1>\n<p>Политика обработки персональных данных (выдержка) Сервис обрабатывает данные пользователей для предоставления доступа к контенту, улучшения рекомендаций и обеспечения безопасности аккаунтов. Данные о просмотрах используются в обезличенном виде для аналитики. Пользователь может запросить сведения об обрабатываемых данных через форму обратной связи. Полный текст политики — на странице filmoskop.ru/privacy.</p>\n</div>"
}
- Final:
{
  "doc_id": 57,
  "title": "Политика обработки персональных данных (выдержка)",
  "category": "Аккаунт и профили",
  "format": "html",
  "raw": "Политика обработки персональных данных (выдержка) Сервис обрабатывает данные пользователей для предоставления доступа к контенту, улучшения рекомендаций и обеспечения безопасности аккаунтов. Данные о просмотрах используются в обезличенном виде для аналитики. Пользователь может запросить сведения об обрабатываемых данных через форму обратной связи. Полный текст политики — на странице filmoskop.ru/privacy."
}

### markdown
- Initial:
{
  "doc_id": 98,
  "title": "Универсальный чек-лист при любой ошибке",
  "category": "Ошибки воспроизведения",
  "format": "markdown",
  "raw": "# Универсальный чек-лист при любой ошибке\n\nЕсли на экране появилась ошибка, пройдите базовые шаги — они решают большинство разовых сбоев:\n\n1. Перезапустите фильм, затем приложение. 2. Перезагрузите устройство. 3. Проверьте, что приложение обновлено до свежей версии. 4. Проверьте, что интернет работает на других сайтах и приложениях. 5. Загляните на страницу статуса сервиса: возможно, идут технические работы.\n\nЕсли ошибка осталась, найдите в справке статью по её коду — там будут шаги для конкретной ситуации.\n"
}
- Final:
{
  "doc_id": 98,
  "title": "Универсальный чек-лист при любой ошибке",
  "category": "Ошибки воспроизведения",
  "format": "markdown",
  "raw": "Если на экране появилась ошибка, пройдите базовые шаги — они решают большинство разовых сбоев: 1. Перезапустите фильм, затем приложение. 2. Перезагрузите устройство. 3. Проверьте, что приложение обновлено до свежей версии. 4. Проверьте, что интернет работает на других сайтах и приложениях. 5. Загляните на страницу статуса сервиса: возможно, идут технические работы. Если ошибка осталась, найдите в справке статью по её коду — там будут шаги для конкретной ситуации."
}

## Results
Normalized corpus:  1_corpus.json
Normalized queries: 1_queries.json

# Этап 2. Чанкинг и эмбеддинги


## Install
pip install tiktoken razdel
pip install sentence_transformers numpy 


## Chunking strategies

Средняя длина раздела - 795 символов. Кириллический текст обычно токенизируется по 1,8–2,5 символа на токен.
Поэтому размер чанка в 400 токенов позволит сохранить целиком большую часть документов. 
В то же время, это позволит не размывать вектора для длинных документов.
Также это безопасный размер для моделей с лимитом в 512 токенов.
Сдвиг выбран как 10% от размера чанков.

1. 400 chars with overlap 
  - со сдвигом: chunk_size = 40
  - within text items

2. sentence based 
  - within text items 
  - max_tokens=400

In [164]:
'''
2.0 Load Docs
'''
import json

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)

print("Разделов:", len(docs))

Разделов: 139


In [165]:
'''
2.2 Chunking
'''
import tiktoken
from razdel import sentenize  # сегментация русского текста на предложения

ENC = tiktoken.get_encoding("cl100k_base")

def fixed_size_chunks(text, chunk_size=512, overlap=50):
    """Нарезает текст fixed-size-чанкингом: окно из chunk_size токенов со сдвигом chunk_size − overlap."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        window = tokens[start:start + chunk_size]
        chunks.append(ENC.decode(window))
        # Сдвигаем окно: на overlap токенов меньше длины окна
        start += chunk_size - overlap
    return chunks


def sentence_chunks(text, max_tokens=512):
    """Нарезает текст sentence-based-чанкингом: набирает целые предложения, пока не упрётся в бюджет max_tokens."""
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    current = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(ENC.encode(sent))
        # Следующее предложение не влезает — закрываем чанк
        if current and current_tokens + sent_tokens > max_tokens:
            chunks.append(" ".join(current))
            current = []
            current_tokens = 0
        current.append(sent)
        current_tokens += sent_tokens
    if current:
        chunks.append(" ".join(current))
    return chunks


def describe(name, chunks):
    """Печатает сводку по списку чанков: количество, средний размер, текст на стыке соседних чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    avg = sum(sizes) / len(sizes)
    end0 = chunks[0][-24:].replace("\n", " ")
    start1 = chunks[1][:24].replace("\n", " ")
    print(f"{name}")
    print(f"  чанков: {len(chunks)}, средний размер: {avg:.0f}")
    print(f"  размеры, токенов: {sizes}")
    print(f"  чанк 1 кончается на: ...{end0!r}")
    print(f"  чанк 2 начинается с: {start1!r}...")
    print()

In [166]:
'''
2.2.1 Chunking - execute
'''
chunks_fixed = []
chunks_sentence = []
for i in docs:
  fixed = fixed_size_chunks(i['raw'], chunk_size=400, overlap=40)
  chunks_fixed.extend(fixed)
  
  sentence = sentence_chunks(i['raw'], max_tokens=400)
  chunks_sentence.extend(sentence)

# docs as a list
texts = [item["raw"] for item in docs]
texts_str = ", ".join(texts)

print(f"Корпус: {len(ENC.encode(texts_str))} токена\n")
describe("fixed-size (400/40)", chunks_fixed)
describe("sentence-based (400)", chunks_sentence)

Корпус: 48026 токена

fixed-size (400/40)
  чанков: 209, средний размер: 242
  размеры, токенов: [181, 400, 123, 177, 400, 50, 400, 142, 400, 110, 184, 202, 165, 177, 183, 197, 195, 355, 185, 216, 143, 400, 350, 160, 400, 377, 17, 293, 400, 126, 189, 400, 55, 400, 126, 400, 141, 150, 196, 400, 122, 159, 195, 400, 104, 400, 50, 400, 123, 222, 400, 123, 149, 319, 400, 72, 257, 400, 175, 254, 146, 177, 400, 72, 150, 226, 400, 176, 218, 156, 400, 74, 400, 284, 400, 365, 5, 400, 120, 400, 212, 400, 82, 400, 381, 21, 400, 144, 208, 170, 400, 124, 400, 207, 140, 282, 143, 176, 273, 400, 92, 159, 338, 194, 204, 207, 400, 367, 7, 266, 385, 25, 382, 22, 400, 83, 159, 280, 400, 120, 400, 113, 400, 400, 87, 186, 222, 400, 400, 55, 181, 400, 100, 393, 33, 157, 181, 212, 400, 54, 135, 200, 102, 400, 292, 303, 400, 70, 400, 131, 212, 137, 184, 168, 271, 305, 400, 92, 400, 340, 399, 39, 207, 400, 375, 15, 26, 213, 221, 174, 362, 2, 215, 240, 400, 284, 400, 90, 203, 400, 340, 400, 124, 400, 290, 177, 2

In [167]:
'''
2.3 Chunking Statistics
'''

import statistics  # Медиана и другая базовая статистика

MIN_TOKENS = 100   # Чанк короче этого порога — подозрительно короткий


def is_truncated(chunk):
    """Возвращает True, если чанк не заканчивается завершающим знаком препинания, — значит, оборван."""
    if chunk.rstrip().endswith((".", "!", "?", "…")):
      return False
    else: 
      # print('>>> Truncated: ', chunk)
      return True


def evaluate(name, chunks):
    """Считает метрики качества разбивки: распределение размеров, число коротких и оборванных чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    short = sum(size < MIN_TOKENS for size in sizes)
    truncated = sum(is_truncated(c) for c in chunks)
    print(f"{name}:")
    print(f"  чанков: {len(chunks)}")
    print(f"  размеры, токенов: {sorted(sizes)}")
    print(f"  min / медиана / max: {min(sizes)} / "
          f"{statistics.median(sizes):.0f} / {max(sizes)}")
    print(f"  короче {MIN_TOKENS} токенов: {short}")
    print(f"  обрезаны посреди предложения: {truncated}")

    # for c in chunks:
    #   if len(ENC.encode(c))  < MIN_TOKENS:
    #     print('>>> small size: ', c)
    #    if is_truncated(c):
    #     print('>>> truncated: ', c)


evaluate("fixed-size (400/40)", chunks_fixed)
print()
evaluate("sentence-based (400)", chunks_sentence)

fixed-size (400/40):
  чанков: 209
  размеры, токенов: [2, 5, 7, 15, 17, 21, 22, 22, 25, 26, 33, 39, 40, 45, 50, 50, 54, 55, 55, 55, 70, 72, 72, 74, 82, 83, 87, 90, 92, 92, 97, 100, 102, 104, 110, 113, 120, 120, 122, 123, 123, 123, 124, 124, 126, 126, 131, 131, 135, 137, 140, 141, 142, 143, 143, 144, 146, 149, 150, 150, 156, 157, 159, 159, 159, 160, 161, 163, 164, 165, 168, 170, 174, 174, 175, 176, 176, 177, 177, 177, 177, 181, 181, 181, 183, 184, 184, 185, 186, 189, 194, 195, 195, 196, 197, 199, 200, 202, 203, 204, 207, 207, 207, 208, 212, 212, 212, 213, 215, 216, 218, 221, 222, 222, 226, 240, 254, 257, 266, 271, 273, 278, 279, 280, 282, 284, 284, 290, 292, 293, 303, 305, 319, 338, 340, 340, 350, 354, 355, 362, 365, 367, 375, 377, 381, 382, 382, 385, 393, 399, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400, 400

In [168]:
'''
2.4 Save chunks to files - sentence_based
'''
chunks_final = []
id = 0
for i in docs:
  chunks = sentence_chunks(i['raw'], max_tokens=400)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_final.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks.json", "w", encoding="utf-8") as f:
  json.dump(chunks_final, f, ensure_ascii=False, indent=4)

In [169]:
'''
2.4.1 Save chunks to files - fixed
'''
chunks_fixed = []
id = 0
for i in docs:
  chunks = fixed_size_chunks(i['raw'], chunk_size=400, overlap=40)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_fixed.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks_fixed.json", "w", encoding="utf-8") as f:
  json.dump(chunks_fixed, f, ensure_ascii=False, indent=4)

In [170]:
'''
2.5 Load chunks for Embedding - sentence_based
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]
metas = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks]

print("Разделов:", len(chunks))

Разделов: 195


In [171]:
'''
2.5.1 Load chunks for Embedding - fixed
'''
import json

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("2_chunks_fixed.json", encoding="utf-8") as f:
  chunks_fixed = json.load(f)

texts_fixed = [c["text"] for c in chunks_fixed]
metas_fixed = [{k: c[k] for k in ("id", "doc_id", "category", "format", "title")}
         for c in chunks_fixed]

print("Разделов:", len(chunks_fixed))

Разделов: 209


In [173]:
'''
2.6 Embed chunks - sentence-based
'''
import numpy as np
from sentence_transformers import SentenceTransformer

def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
corpus_vecs = np.ascontiguousarray(embed_passages(embedder, texts), dtype="float32")
np.save("2_kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Чанков: 195, эмбеддинги: (195, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


In [174]:
'''
2.6.1 Embed chunks - fixed
'''
import numpy as np

corpus_vecs_fixed = np.ascontiguousarray(embed_passages(embedder, texts_fixed), dtype="float32")
np.save("2_kb_vecs_fixed.npy", corpus_vecs_fixed)   

print(f"Чанков: {len(texts_fixed)}, эмбеддинги: {corpus_vecs_fixed.shape}")
print("Пример метаданных:", metas_fixed[1])

Чанков: 209, эмбеддинги: (209, 768)
Пример метаданных: {'id': 1, 'doc_id': 1, 'category': 'Функции и контент', 'format': 'text', 'title': 'Синхронизация между устройствами'}


# Этап 2. Чанкинг и эмбеддинги - Report


# statistics
fixed-size (400/40):
-  чанков: 209
-  min / медиана / max: 2 / 212 / 400
-  короче 100 токенов: 31
-  ** обрезаны посреди предложения: 55 **

sentence-based (400):
-  чанков: 195
-  min / медиана / max: 26 / 221 / 404
-  короче 100 токенов: 23
-  ** обрезаны посреди предложения: 0 **


## стратегия эмбединга
Далее используется sentence-based embedding - ввиду того, что он не разбивает на части предложения.

Ручной просмотр чанков выполнен выборочно.


## Results
2_chunks.json
2_kb_vecs.npy

# Этап 3. Поисковая инфраструктура и retrieval

## Vector DB
В качестве хранилища выбрано Chroma DB.
- для удобства хранения и поиска по индексу в БД
- для возможности поиска внутри Категорий (и другим метаданным)
- Qdrant - более требовательная к ресурсам технология.

При переходе в промышленную эксплуатацию предполагается перейти на Qdrant - для удобства сопровождения и эксплуатации.


# install
pip install chromadb

In [175]:
'''
3.1 Load vectors to DB - sentence-based
'''
import chromadb
from sentence_transformers import SentenceTransformer

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass

col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col.add(
    ids=[str(i) for i in range(len(texts))],             # Уникальные id
    embeddings=corpus_vecs.tolist(),                     # Векторы чанков
    documents=texts,                                     # Сами тексты
    metadatas=metas,                                     # Метаданные
)
print("Записей в коллекции:", col.count())

Записей в коллекции: 195


In [176]:
'''
3.1.1 Load vectors to DB - fixed
'''
try:
    client.delete_collection("knowledge_base_fixed")           # Чистим прошлый прогон
except Exception:
    pass

col_fixed = client.create_collection(
    name="knowledge_base_fixed",
    configuration={"hnsw": {"space": "cosine"}})         # Косинусная близость

col_fixed.add(
    ids=[str(i) for i in range(len(texts_fixed))],             # Уникальные id
    embeddings=corpus_vecs_fixed.tolist(),                     # Векторы чанков
    documents=texts_fixed,                                     # Сами тексты
    metadatas=metas_fixed,                                     # Метаданные
)
print("Записей в коллекции:", col_fixed.count())

Записей в коллекции: 209


In [177]:
'''
3.1.1 Test 1 search
Correct doc: #140
'''
from sentence_transformers import SentenceTransformer

QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.162 [id: 112] [doc_id: 108] [title: Родительский контроль: обзор] Родительский контроль — общее название для настроек, которые ограничивают то, что ребёнок видит и делает в онлайн-кинотеатре. Почти везде он складывается из двух частей: фильтра по возрастным рейтингам и ограничения платных действий. Возрастной рейтинг присваивается фильму при выдаче прокатного удостоверения, сервис только показывает его в карточке и использует как фильтр. Рейтинг относится к фильму целиком, поэтому добрый на вид мультфильм с одной тревожной сценой может стоять на ступень выше, чем ожидают родители. Ограничение платных действий закрывает вторую типичную неприятность — оплату случайным нажатием. На телевизоре это особенно легко: карточка новинки и кнопка с ценой находятся в двух движениях пульта. Сервисы решают это по-разному: одни убирают раздел с покупками из детского режима, другие просят подтверждение перед оплатой.
  dist=0.169 [i

In [178]:
'''
3.2 Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

# preliminary selection
THRESHOLD = 0.83
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [191]:
'''
3.2.1 Test: 3 queries 
'''
import json

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]


for qi, q in enumerate(queries[:3]):
  ret = retrieve_dict(q["query"])
  print(f"\n[{qi}] {q['query']} {q['relevant_docs']}")
  if not ret:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for i in ret:
    print(f"    sim={i['similarity']:.3f} #{i['doc_id']} {i['text'][:58]} #{i['id']}")


[0] что означает ошибка P-102 [85]
    sim=0.870 #85 Ошибка P-102 появляется, когда число одновременных просмот #92
    sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает св #191
    sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить #146
    sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает  #147
    sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена у #137

[1] какая версия webOS или Tizen нужна для приложения на телевизоре [146]
    sim=0.878 #146 Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года),  #157
    sim=0.838 #129 Ошибка P-105 появляется, когда на устройстве установлена у #137
    sim=0.836 #172 Приложение на телевизоре обновляется само, если в магазине #183
    sim=0.836 #139 Если вы не знаете, к какой платформе относится ваш телевиз #148
    sim=0.834 #176 Если приложение на телевизоре закрывается само или зависае #189

[2] обязателен ли HDCP 2.2 для просмотра в 4K [110]
    sim=

In [192]:
'''
3.3 Test: compare with relevant docs
'''
relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    ret = retrieve_dict(q["query"])
    # hits = retrieve(q["query"], k=5, threshold=0.0)
    top_ids = [i['doc_id'] for i in ret]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    ret = retrieve_dict(q["query"], k=5, threshold=0.75)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for i in ret:
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} sim={i['similarity']:.3f} #{i['id']} {i['text'][:60]}")

Проблемные запросы: [3, 10, 13, 15, 18, 19, 20, 22, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39, 41, 43, 44, 46, 47, 48, 50, 51, 53, 54]

[3] как подключить саундбар по HDMI ARC  (релевантные id: [22])
      sim=0.842 #133 Если при воспроизведении звук идет, а изображения нет — экра
      sim=0.834 #77 Объёмный звук дома собирают двумя способами, и выбор между н
    + sim=0.816 #24 Обозначение 5.1 описывает раскладку каналов: пять основных к
      sim=0.812 #56 Встроенные динамики телевизора делают тонкими вместе с корпу
      sim=0.810 #160 У вас уже есть всё для домашнего кинотеатра — просто вы назы

[10] сколько профилей можно создать в одном аккаунте  (релевантные id: [63])
      sim=0.893 #70 Сколько профилей помещается в один аккаунт и как их создать,
    + sim=0.874 #71 Профиль — это отдельное пространство внутри одного аккаунта:
      sim=0.868 #69 Аккаунт и профиль — не одно и то же. Аккаунт заводится на но
    + sim=0.854 #72 Если человек пользуется сервисом изредка, п

In [193]:
'''
3.4 check recall@5, precision@5, MRR
'''
import numpy as np

def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

def precision_at_k(found_ids, relevant_ids, k=5):
  """Доля релевантных среди k выданных чанков."""
  hits = [i for i in found_ids[:k] if i in relevant_ids]
  return len(hits) / k

def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

def basic_search(query, k=5, threshold=0.8, col=col):
  ret = retrieve_dict(query, col=col, k=k, threshold=threshold)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.20   1.00   0.33
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   1.00
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   0.50
на каком тарифе доступен звук 5.1              2   0.40   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [194]:
'''
3.4.1 check recall@5, precision@5, MRR - fixed
'''
import numpy as np

K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"], col=col_fixed)
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Basic Search
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                      1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при     1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K      1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC            1   0.40   1.00   0.33
что означает ошибка P-109                      1   0.20   1.00   1.00
что значит ошибка P-111                        1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него      1   0.40   1.00   0.50
сколько дней длится пробный период             1   0.40   1.00   1.00
какая скорость интернета нужна для 4K          1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend     1   0.20   1.00   1.00
сколько профилей можно создать в одном акк     1   0.40   1.00   0.33
на каком тарифе доступен звук 5.1              2   0.40   0.50   1.00
сколько устройств можно привязать к аккаун     1   0.20   1.00   1.00
ско

In [195]:
'''
3.5 analize K & threshhold
'''
for tr in [0.7, 0.75, 0.8, 0.83]:
  for K in [1, 3, 5, 10, 20]:
    rows = []
    for q in queries:
      rel = set(q["relevant_docs"])
      found = basic_search(q["query"], k=K, threshold=tr)
      rows.append(recall_at_k(found, rel, K))

    print(f"threshold = {tr} | K = {K} | recall@{K} = {np.mean(rows):.3f}")

threshold = 0.7 | K = 1 | recall@1 = 0.384
threshold = 0.7 | K = 3 | recall@3 = 0.679
threshold = 0.7 | K = 5 | recall@5 = 0.768
threshold = 0.7 | K = 10 | recall@10 = 0.911
threshold = 0.7 | K = 20 | recall@20 = 0.973
threshold = 0.75 | K = 1 | recall@1 = 0.384
threshold = 0.75 | K = 3 | recall@3 = 0.679
threshold = 0.75 | K = 5 | recall@5 = 0.768
threshold = 0.75 | K = 10 | recall@10 = 0.911
threshold = 0.75 | K = 20 | recall@20 = 0.973
threshold = 0.8 | K = 1 | recall@1 = 0.384
threshold = 0.8 | K = 3 | recall@3 = 0.661
threshold = 0.8 | K = 5 | recall@5 = 0.732
threshold = 0.8 | K = 10 | recall@10 = 0.875
threshold = 0.8 | K = 20 | recall@20 = 0.938
threshold = 0.83 | K = 1 | recall@1 = 0.366
threshold = 0.83 | K = 3 | recall@3 = 0.580
threshold = 0.83 | K = 5 | recall@5 = 0.598
threshold = 0.83 | K = 10 | recall@10 = 0.670
threshold = 0.83 | K = 20 | recall@20 = 0.696


# Этап 3. Поисковая инфраструктура и retrieval - Report

Требования: recall@5 >= 0,85

На основе анализа выбраны параметры:
Threshold = 0.75 - max, не удовлетворяет требованиям


## chunking strategy
(threshold = 0.83)
- fixed size
precision@5 = 0.243
recall@5    = 0.732
MRR         = 0.590

- sentence-based
precision@5 = 0.250
recall@5    = 0.732
MRR         = 0.615

** Метрики для sentence-based и fixed size chunking почти совпадают **


## llama.cpp

### GPU
sudo docker run -d --gpus all -p 8080:8080 --name llm ghcr.io/ggml-org/llama.cpp:server-cuda \
  -hf unsloth/Qwen3.5-4B-GGUF:Q4_K_M --host 0.0.0.0 --port 8080 -ngl 99 -c 4096

### CPU
sudo docker run -d -p 8080:8080 --name llm ghcr.io/ggml-org/llama.cpp:server \
  -hf unsloth/Qwen3.5-4B-GGUF:Q4_K_M --host 0.0.0.0 --port 8080 -c 4096

sudo docker start llm 

curl http://localhost:8080/health
sudo docker logs -f llm

# Этап 4. Диагноз и улучшение retrieval

# install
```
pip install openai
```

In [196]:
'''
4.0 Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.75
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
'''
4.1 Reranking
'''
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="mps")

def retrieve_rerank(query, k=5):
  ret = retrieve_dict(query, k=20)
  cands = [i['text'] for i in ret]
  scores = reranker.predict([(query, c) for c in cands])
  for i,s in enumerate(scores, 0):
    ret[i]['score'] = s
  sorted_ret = sorted(ret, key=lambda x: x["score"], reverse=True)
  return sorted_ret[:k]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
'''
4.1.1
'''
import numpy as np

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)
corpus = [i['raw'] for i in docs]

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)
queries = [i for i in queries if i['kind'] != 'gap']
queries_txt = [i['query'] for i in queries if i['kind'] != 'gap']


rows = []
for q in queries:
  rel = set(q["relevant_docs"])
  found = retrieve_rerank(q["query"])
  ids = [i['doc_id'] for i in found]
  rows.append({
      "n_rel": len(rel),
      "p": precision_at_k(ids, rel, K),
      "r": recall_at_k(ids, rel, K),
      "rr": reciprocal_rank(ids, rel),
  })

print('>>> Reranking')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{q['kind']}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

>>> Reranking
запрос                                       |R|    P@5    R@5     RR
что означает ошибка P-102                   exact   1   0.20   1.00   1.00
какая версия webOS или Tizen нужна для при  exact   1   0.20   1.00   1.00
обязателен ли HDCP 2.2 для просмотра в 4K   exact   1   0.40   1.00   1.00
как подключить саундбар по HDMI ARC         exact   1   0.20   1.00   1.00
что означает ошибка P-109                   exact   1   0.20   1.00   1.00
что значит ошибка P-111                     exact   1   0.20   1.00   1.00
сколько стоит тариф Максимум и что в него   exact   1   0.40   1.00   1.00
сколько дней длится пробный период          exact   1   0.40   1.00   1.00
какая скорость интернета нужна для 4K       exact   1   0.20   1.00   1.00
есть ли приложение Фильмоскопа для Nintend  exact   1   0.20   1.00   1.00
сколько профилей можно создать в одном акк  exact   1   0.40   1.00   1.00
на каком тарифе доступен звук 5.1           exact   2   0.60   1.00   1.00
сколько устройст

In [ ]:
import numpy as np

for qi in range(len(queries)):
    reranked = retrieve_rerank(queries[qi]['query'])
    rel = queries[qi]['relevant_docs']
    print(f"\nЗапрос: {query}")
    print(f"  После реранкинга top-5:")
    for rank, i in enumerate(reranked, 1):
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} {rank}. #{i['doc_id']:03d} {i['text'][:70]}...")


def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0


Запрос: какая гарантия на холодильник
  Биэнкодер top-5 (до реранкинга):
  После реранкинга top-5:
    + 1. #085 Ошибка P-102 появляется, когда число одновременных просмотров на аккау...
      2. #165 Ошибка P-112 появляется, когда аккаунт заблокирован за нарушение польз...
      3. #090 Ошибка P-110 появляется, когда происходит сбой на стороне сервиса: с в...
      4. #094 Ошибка P-101 появляется, когда приложению не удаётся соединиться с сер...
      5. #102 Если просмотр прерывается, плеер показывает короткий код вида P-1xx. Б...

Запрос: какая гарантия на холодильник
  Биэнкодер top-5 (до реранкинга):
  После реранкинга top-5:
    + 1. #146 Телевизоры Smart TV: Samsung (Tizen, модели с 2018 года), LG (webOS 4....
      2. #172 Приложение на телевизоре обновляется само, если в магазине включено ав...
      3. #139 Если вы не знаете, к какой платформе относится ваш телевизор, найдите ...
      4. #040 Фильмы в онлайн-кинотеатрах передаются в зашифрованном виде. Это услов...
      5.

In [139]:
# import numpy as np

# for qi in range(3):
#     query = queries_txt[qi]
#     cand = retrieve_dict(query_vecs[qi], doc_vecs, n=20)
#     reranked = rerank(query, cand, CORPUS, reranker, k=5)
#     rel = RELEVANT[qi]
#     print(f"\nЗапрос: {query}")
#     print(f"  Биэнкодер top-5 (до реранкинга):")
#     for rank, idx in enumerate(cand[:5], 1):
#         mark = "+" if idx in rel else " "
#         print(f"    {mark} {rank}. #{idx:03d} {CORPUS[idx][:70]}...")
#     print(f"  После реранкинга top-5:")
#     for rank, idx in enumerate(reranked, 1):
#         mark = "+" if idx in rel else " "
#         print(f"    {mark} {rank}. #{idx:03d} {CORPUS[idx][:70]}...")

In [117]:
'''
4.0.1 load queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]

In [26]:
'''
4.1 LLM Generate
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [27]:
'''
4.2 Expansion and Ranking
'''
from collections import defaultdict

def expand(query, n=5):
  """Query expansion: n перефразировок запроса официальными словами базы."""
  text = generate([
      {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
        f"официальными словами из базы знаний. Используй полные фразы (не одно "
        f"слово), синонимы и термины. По одной формулировке на строку, без "
        f"нумерации.\n\nЗапрос: {query}"}])
  expanded = [l.strip() for l in text.splitlines() if l.strip()][:n]
  # print('\n>>> expand: ', query, '\n', expanded)
  return expanded

def rrf(ranked_lists, k=60, top_k=5):
  """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
  scores = defaultdict(float)
  for lst in ranked_lists:
      for rank, doc in enumerate(lst, start=1):
          scores[doc] += 1.0 / (k + rank)
  ranked = sorted(scores.items(), key=lambda x: -x[1])
  return [doc for doc, _ in ranked[:top_k]]

In [28]:
'''
4.2.1 RRF Test on 3 queries
'''
for qi in [22, 30, 33]:
  query = queries[qi]
  rel = set(queries[qi]["relevant_docs"])

  # basic search
  basic = retrieve_dict(query['query'])
  # print('>>> basic: ', basic)

  # expanded search
  paras = expand(query['query'])
  # print('>>> paras: ', paras)
  exp = [retrieve_dict(p) for p in paras]
  exp.append(basic)

  tot = []
  for q in exp:
    r = [(i['id'], i['text'], i['similarity'], i['doc_id']) for i in q]
    tot.append(r)
  ranked = rrf(tot)
  print('>>> ranked: ', ranked)

  mark = lambda i: "+" if i in rel else " "
  print(f"\n[{qi}] {query['query']} {query['relevant_docs']}")
  
  for p in paras:
    print("   ·", p)

  print("  обычный поиск top5:",
    [f"{mark(i['doc_id'])}#{i['doc_id']}/{i['id']} {i['text'][:32]}" for i in basic])
  print("  query expansion top5:",
    [f"{mark(i[0])}#{i[0]} {i[1][:32]}" for i in ranked])

>>> ranked:  [(85, 'Раздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.', 0.8238, 74), (117, 'Такие обращения направляйте на help@filmoskop.ru с пометкой о том, что вопрос от организации, — письмо передадут нужной команде.', 0.8421, 103), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после паузы; чтобы позиция точно сохранилась, поставьте видео на паузу перед закрытием приложения. Позиция хранится отдельно для каждого профиля.', 0.8258, 111), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после па

In [29]:
'''
4.3 
'''
def basic_search(query):
  ret = retrieve_dict(query)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids


def precision_at_k(found_ids, relevant_ids, k):
    """Доля релевантных среди k выданных чанков."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / k


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

In [30]:
'''
4.3.1 Retrieve func
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8
K = 10

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")

def retrieve(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18043.33it/s]


In [31]:
'''
4.4 HyDE
'''
def retrieve_hyde(query, answer, col, embedder, k=10, threshold=0.0):
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out


def retrieve_hyde_dict(query, answer, k=10, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    ) 
  return out


def hyde(query, top_k=10):
    """HyDE: ищем по среднему векторов запроса и гипотетического ответа."""
    doc = generate([
        {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                      "онлайн-кинотеатра."},
        {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
         f"ответ из базы знаний на вопрос клиента официальным языком. Только "
         f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
    hits = retrieve_hyde(query, doc, col, embedder, top_k)
    return doc, hits


for qi in [22, 30, 33]:
    query = queries[qi]["query"]
    rel = set(queries[qi]["relevant_docs"])
    doc, hits = hyde(query)
    mark = lambda i: "+" if i in rel else " "

    found = retrieve_dict(query)
    # print('>>> found: ', found)

    print(f"\n[{qi}] {query}")
    print("  гипотетический ответ:", doc)
    print("  обычный поиск top5:",
        [f"{mark(i['doc_id'])}#{i['doc_id']} {i['text'][:32]}" for i in found])
    print("  HyDE top5:", [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in hits]) 


[22] начал в метро с телефона, дома хочу досмотреть на большом экране с той же секунды
  гипотетический ответ: Для просмотра фильма с того же места, где вы начали его смотреть в метро, необходимо создать аккаунт и авторизоваться на устройстве дома. После входа в систему вы сможете продолжить просмотр с точки останова.
  обычный поиск top5: [' #24 # Какой тариф подойдёт именно ва', ' #144 Ограничения: **скачивать можно н', ' #1 Переносить их некуда, поэтому на', ' #44 Кино давно смотрят не с одного э', ' #150 # Игры и кино на одной консоли У', ' #109 Офлайн-просмотр — это фильмы и с', ' #113 # Тарифы «Фильмоскопа» Тариф выб', ' #82 Сам роутер лучше держать в откры', ' #74 Раздел в разработке Мы готовим э', ' #99 Совместный просмотр Функция расс']
  HyDE top5: [' #169 Ограничения: **скачивать можно н', ' #111 Совместный просмотр Функция расс', '+#130 Позиция просмотра сохраняется ав', ' #174 # Игры и кино на одной консоли У', ' #23 После возвращения домой каталог ', ' #132 # Тарифы «Фил

In [32]:
'''
4.5 Estimate
'''
import numpy as np

K = 10

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries[:]:
    query, rel = q["query"], set(q["relevant_docs"])

    # Basic
    base = retrieve_dict(query)
    base_docs = [t['doc_id'] for t in base]
    rows["Обычный поиск"].append(recall_at_k(base_docs, rel, K))

    # Expand
    exp = rrf([retrieve(query)] + [retrieve(p) for p in expand(query)])
    exp_docs = [t[0] for t in exp]
    rows["Query expansion"].append(recall_at_k(exp_docs, rel, K))

    # HyDE
    _, hyd = hyde(query, K)
    hyd_docs = [t[0] for t in hyd]
    rows["HyDE"].append(recall_at_k(hyd_docs, rel, K))

n = len(queries)
print(f"{'Подход':<18}{'recall@{K}':>10}")
for name in rows:
  print(f"{name:<18}{np.mean(rows[name]):>10.3f}")

Подход            recall@{K}
Обычный поиск          0.875
Query expansion        0.634
HyDE                   0.938


# Этап 4. Диагноз и улучшение retrieval - Report

Реализованные варианты улучшения:
- Query Expansion
- HyDE - показал хороший результат - рекомендуерся включить в пилот

Для дальнейшего улучшения наиболее перспективным является "реранкер cross-encoder" - к сожалению, не успел реализовать.

# Этап 5. Генерация ответов

In [33]:
'''
5.0 load all queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

In [34]:
'''
5.1 Pipeline
'''
SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def build_messages(query, chunk_dicts):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    # print('>>> chunk_dicts: ', chunk_dicts)
    context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(chunk_dicts, 1))
    user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def rag_answer(query, search_fn, k=10):
    """Полный RAG: retrieval -> промпт -> генерация."""
    ret = search_fn(query, k=k)
    msg = build_messages(query, ret)
    answer = generate(msg)
    return answer, [i['id'] for i in ret], [i['text'] for i in ret]

In [36]:
'''
5.2 Test different kinds of questions
'''
print('>>> exact queries')
for q in [x for x in queries if x["kind"] == "exact"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> semantic queries')
for q in [x for x in queries if x["kind"] == "semantic"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> multi queries')
for q in [x for x in queries if x["kind"] == "multi"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> gaps')
for q in [x for x in queries if x["kind"] == "gap"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

>>> exact queries

В: что означает ошибка P-102
   чанки: [98, 207, 159, 160, 150, 54, 60, 102, 106, 191]  (['# Ошибка P-102: превышен лимит про', 'Ошибка P-109 появляется, когда на ', 'Ошибка P-106 появляется, когда пле', 'Ошибка P-104 появляется, когда уст', 'Ошибка P-105 появляется, когда на ', 'Ошибка P-103: контент недоступен в', 'Ошибка P-113 появляется, когда фай', 'Ошибка P-110 появляется, когда про', 'Ошибка P-101 появляется, когда при', 'Ошибка P-112 появляется, когда акк'])
   О: Ошибка P-102 появляется, когда число одновременных просмотров на аккаунте превышает лимит тарифа. Для решения проблемы нужно проверить количество активных экранов, обновить приложение и при необходимости перейти на тариф с большим лимитом.

В: какая версия webOS или Tizen нужна для приложения на телевизоре
   чанки: [171, 99, 150, 199, 161, 205, 86, 200, 162, 34]  (['# На каких устройствах работает «Ф', '# Поддерживаемые устройства На как', 'Ошибка P-105 появляется, когда на ', 'Приложение на телеви

# Этап 5. Генерация ответов - Report

Результат: RAG-система, которая принимает вопрос и возвращает ответ или корректный отказ.

# Этап 6. Оценка качества и диагностика

## install
pip install pandas

In [37]:
'''
6.0 Load judge_cases
'''
import json

with open("6_judge_cases.json", encoding="utf-8") as f:
  judge_cases = json.load(f)

with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]

In [38]:
'''
6.1 Judge Conf
'''
JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке справочных ответов. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Соответствие вопросу не оценивай — только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — добавлено только безобидное обобщение;
3 — добавлена неподтверждённая деталь;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» ничего не утверждает и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с полями reason и score."""

REL_PROMPT = """Оцени РЕЛЕВАНТНОСТЬ (answer relevance):
отвечает ли ответ на заданный вопрос по существу.
Правдивость не проверяй — только соответствие вопросу.

Шкала:
5 — прямой и полный ответ;
4 — ответ по существу с небольшими отступлениями;
3 — закрыта только часть вопроса;
2 — текст на нужную тему, но ответа нет;
1 — другая тема или отказ отвечать.

Вопрос: {question}

Ответ: {answer}

Верни JSON с полями reason и score."""

JUDGE_RESPONSE_FORMAT = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {
            "reason": {"type": "string", "maxLength": 240},
            "score": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
        "required": ["reason", "score"],
        "additionalProperties": False,
    },
}


def judge(prompt):
    """Возвращает оценку 1–5 и обоснование LLM-судьи."""
    response = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=200,
        response_format=JUDGE_RESPONSE_FORMAT,
        extra_body=NO_THINK,
    )
    value = json.loads(response.choices[0].message.content)
    score = value.get("score")
    reason = value.get("reason")
    if type(score) is not int or not 1 <= score <= 5:
        raise ValueError(f"Некорректная оценка судьи: {score!r}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Судья не вернул обоснование")
    return score, reason.strip()

# кортеж «(оценка, обоснование)»
def judge_faithfulness(context, answer):
    """Оценивает верность ответа контексту."""
    prompt = FAITH_PROMPT.format(context=context, answer=answer)
    return judge(prompt)

# кортеж «(оценка, обоснование)»
def judge_relevance(question, answer):
    """Оценивает соответствие ответа вопросу."""
    prompt = REL_PROMPT.format(question=question, answer=answer)
    return judge(prompt)

def build_context(chunk_ids):
    """Склеивает чанки в нумерованный контекст."""
    return "\n".join(f"[{n}] {texts[i]}" for n, i in enumerate(chunk_ids, 1))

In [39]:
'''
6.3 Check Cases
'''
for case in judge_cases:
  print('>>> case: ', case['query'])
  context = build_context(case["context_ids"])
  print(judge_faithfulness(context, case["answer"]))
  print(judge_relevance(case["query"], case["answer"]))
  print('\n')

>>> case:  что означает ошибка P-102
(5, 'Ответ содержит все ключевые факты из фрагмента [1]: причина ошибки (превышение лимита одновременных просмотров), необходимость проверки количества экранов, обновление приложения и переход на тариф с большим лимитом. Дополнительные детали (у')
(5, 'Ответ полностью соответствует вопросу, объясняя причину возникновения ошибки P-102 (превышение лимита просмотров) и предоставляя конкретные шаги по её устранению. Информация является прямой и исчерпывающей для данного запроса.')


>>> case:  какая версия webOS или Tizen нужна для приложения на телевизоре
(5, 'Утверждения в ответе полностью соответствуют информации, содержащейся во фрагментах [1] и [2]. Фрагменты явно указывают, что для Samsung требуется Tizen с моделями с 2018 года, а для LG — webOS 4.0 и новее. Никаких фактов не добавлено, иска')
(5, 'Ответ содержит прямую и конкретную информацию, касающуюся требуемых версий операционных систем (Tizen и webOS) для приложений на телевизорах разных бре

In [40]:
'''
6.4 Определить порог - 3 или 4
'''
import numpy as np

# Четыре параллельных списка: по одному значению на каждый из 10 кейсов.
# judge_* — оценки нашего судьи, human_* — оценки асессора из corpus.json
judge_faith, judge_rel = [], []
human_faith, human_rel = [], []
# judged_rows — оценки обоих судей вместе с обоснованиями,
# чтобы потом напечатать проблемные ответы
judged_rows = []

for case in judge_cases:
    context = build_context(case["context_ids"])
    faith, faith_reason = judge_faithfulness(context, case["answer"])
    rel, rel_reason = judge_relevance(case["query"], case["answer"])
    judge_faith.append(faith)
    judge_rel.append(rel)
    human_faith.append(case["human_faithfulness"])
    human_rel.append(case["human_relevance"])
    judged_rows.append(
        {
            "id": case["id"],
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }
    )


def decision_agreement(scores, human_scores, threshold):
    """Считает совпадения решений по заданному порогу."""
    return sum(
        (score < threshold) == (human < threshold)
        for score, human in zip(scores, human_scores)
    )


print("Средний faithfulness:", round(float(np.mean(judge_faith)), 1))
print("Средний answer relevance:", round(float(np.mean(judge_rel)), 1))
print(
    "Точное совпадение faith:",
    sum(a == b for a, b in zip(judge_faith, human_faith)),
)
print(
    "Точное совпадение relevance:",
    sum(a == b for a, b in zip(judge_rel, human_rel)),
)
for threshold in (3, 4):
    print(
        "Порог ниже", threshold,
        "| faith", decision_agreement(judge_faith, human_faith, threshold),
        "| relevance", decision_agreement(judge_rel, human_rel, threshold),
    )

print("Проблемные ответы по рабочему порогу:")
for row in judged_rows:
    if row["faith"] < 4 or row["rel"] < 3:
        print(row)

Средний faithfulness: 5.0
Средний answer relevance: 3.7
Точное совпадение faith: 2
Точное совпадение relevance: 3
Порог ниже 3 | faith 2 | relevance 3
Порог ниже 4 | faith 2 | relevance 3
Проблемные ответы по рабочему порогу:
{'id': 3, 'faith': 5, 'rel': 1, 'faith_reason': "Ответ 'В базе знаний ответа нет' является корректным, так как ни одно из утверждений в ответе не содержит фактов, которые можно было бы проверить на соответствие предоставленным фрагментам. Фрагменты содержат подробную информацию о тарифах,", 'rel_reason': 'Ответ прямо указывает на отсутствие информации в базе знаний, что делает невозможным提供任何 релевантную информацию по существу вопроса.'}


# Threshold selected as 4

In [41]:
'''
6.5.0
'''
import json

with open("1_corpus.json", encoding="utf-8") as f:
    docs = json.load(f)

print("Разделов:", len(docs), "| Запросов:", len(queries))
print(queries[0])

Разделов: 154 | Запросов: 66
{'query': 'что означает ошибка P-102', 'kind': 'exact', 'relevant_docs': [85], 'gold': 'превышено число одновременных просмотров по тарифу; нужно остановить просмотр на другом устройстве или перейти на тариф с большим лимитом'}


In [42]:
'''
6.5.1
'''
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer


def naive_split(text, size=120):
    """Разрезает text на фрагменты по size символов без перекрытия.

    Возвращает список строк; границы предложений не учитываются.
    """
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks, chunk_doc = [], []                # Тексты чанков и раздел-источник
for d in docs:
    for piece in naive_split(d["raw"]):
        chunks.append(piece)
        chunk_doc.append(d["doc_id"])     # Чанк помнит, из какого раздела вырезан

print("Чанков:", len(chunks))             # Чанков: 108

# Эмбеддер темы: multilingual-e5-base с префиксами query: и passage:
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")
doc_vecs = embedder.encode(["passage: " + c for c in chunks],
                           normalize_embeddings=True, show_progress_bar=False)


def search(query, k=5):
    """Ищет для query ближайшие k чанков.

    Возвращает тексты чанков и идентификаторы разделов-источников.
    """
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    top = np.argsort(q_vec @ doc_vecs.T)[::-1][:k]
    return [chunks[i] for i in top], [chunk_doc[i] for i in top]


# Генерация: тот же локальный сервер и тот же заземляющий промпт
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def generate(messages, max_tokens=200):
    """Отправляет messages локальной модели и ограничивает ответ max_tokens.

    Возвращает очищенный текст ответа; temperature=0 — для воспроизводимости.
    """
    resp = client.chat.completions.create(
        model="qwen3.5", messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()


def rag_answer(query, k=5):
    """Запускает RAG для query по k найденным чанкам.

    Возвращает ответ, тексты чанков и идентификаторы разделов-источников.
    """
    ctx, got_docs = search(query, k)
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
    answer = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
         f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {query}"}])
    return answer, ctx, got_docs

Чанков: 1049


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8395.61it/s]


In [43]:
'''
6.6
'''
import re

JUDGE_FAITH = (
    "Ты — строгий эксперт по оценке ответов. Оцени, насколько ОТВЕТ опирается "
    "только на ФРАГМЕНТЫ. 5 — каждое утверждение ответа есть во фрагментах; "
    "1 — ответ содержит факты, которых во фрагментах нет. Ответ «в базе знаний "
    "ответа нет» считай полностью опирающимся на фрагменты (5). "
    "Верни только одну цифру от 1 до 5.")

JUDGE_RELEV = (
    "Ты — строгий эксперт по оценке ответов. Оцени, отвечает ли ОТВЕТ по "
    "существу на ВОПРОС. 5 — прямой и полный ответ на заданный вопрос; "
    "1 — ответ не по теме или отказ отвечать. Верни только одну цифру от 1 до 5.")

JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")


def judge(question, context, answer, gold):
    """Оценивает answer для question по context и эталонному gold.

    Возвращает faithfulness, answer relevance и бинарную correctness.
    """
    ctx = "\n".join(f"[{i}] {c}" for i, c in enumerate(context, 1))
    faith = generate([{"role": "system", "content": JUDGE_FAITH},
                      {"role": "user", "content":
                       f"ФРАГМЕНТЫ:\n{ctx}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    relev = generate([{"role": "system", "content": JUDGE_RELEV},
                      {"role": "user", "content":
                       f"ВОПРОС: {question}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    correct = generate([{"role": "system", "content": JUDGE_CORRECT},
                        {"role": "user", "content":
                         f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ: {gold}\n\n"
                         f"ОТВЕТ: {answer}"}], max_tokens=8)
    return (int(re.search(r"[1-5]", faith).group()),
            int(re.search(r"[1-5]", relev).group()),
            int(re.search(r"[01]", correct).group()))

In [46]:
'''
6.7
'''
import pandas as pd

# Широкий вывод: иначе pandas свернёт часть столбцов в многоточие
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)


def precision_at_k(got_docs, relevant):
    """Доля слотов выдачи, занятых уникальными релевантными разделами."""
    # Пересечение множеств: два чанка одного раздела дают один релевантный
    # раздел, а знаменатель — размер выдачи, то есть K
    return len(set(got_docs) & set(relevant)) / len(got_docs) if got_docs else 0.0


def recall_at_k(got_docs, relevant):
    """Доля релевантных разделов, которые поиск донёс до промпта."""
    # Знаменатель другой — все релевантные разделы из разметки
    return sum(d in got_docs for d in relevant) / len(relevant) if relevant else 0.0


def mrr(got_docs, relevant):
    """Обратный ранг первого релевантного раздела."""
    for rank, doc in enumerate(got_docs, start=1):
        if doc in relevant:
            return 1 / rank      # Первое место — 1, второе — 0,5, третье — 1/3
    return 0.0                   # Релевантных разделов в выдаче нет вовсе


rows = []

# Прогон по всем 30 запросам: на каждый — ответ системы и три оценки судьи
# Ввиду того, что проект выполнялся локально на CPU, прогон осуществляется по подмножеству запросов
brief = [i for i in queries if i['kind'] == 'exact'][:3] \
  + [i for i in queries if i['kind'] == 'semantic'][:3] \
  + [i for i in queries if i['kind'] == 'multi'][:3] \
  + [i for i in queries if i['kind'] == 'gap'][:3] 

count = 0
for q in brief:
  count += 1
  answer, ctx, got_docs = rag_answer(q["query"], k=10)
  faith, relev, correct = judge(q["query"], ctx, answer, q["gold"])
  rows.append({
      "kind": q["kind"],                # Тип запроса для разбивки по срезам
      "query": q["query"],
      # Метрики поиска считаем по разделам-источникам найденных чанков
      "precision@5": precision_at_k(got_docs, q["relevant_docs"]),
      "recall@5": recall_at_k(got_docs, q["relevant_docs"]),
      "MRR": mrr(got_docs, q["relevant_docs"]),
      # Метрики генерации: их проставил судья
      "faithfulness": faith,
      "answer relevance": relev,
      "correctness": correct,
      "answer": answer,                 # Ответ и контекст сохраняем, чтобы
      "ctx": ctx,                       # разобрать провалы глазами в следующем задании
  })
  # print('>>> # ', count, q)

report = pd.DataFrame(rows)

# Только числовые столбцы: по ним и считаем средние
METRICS = ["precision@5", "recall@5", "MRR", "faithfulness",
           "answer relevance", "correctness"]

print("Сводный профиль:")
print(report[METRICS].mean().round(3))          # Средние по всему набору
print("\nПо типам запросов:")
print(report.groupby("kind")[METRICS].mean().round(3))   # Срезы по полю kind

Сводный профиль:
precision@5         0.083
recall@5            0.667
MRR                 0.370
faithfulness        5.000
answer relevance    4.667
correctness         0.750
dtype: float64

По типам запросов:
          precision@5  recall@5    MRR  faithfulness  answer relevance  correctness
kind                                                                               
exact           0.100     1.000  0.833           5.0             5.000        1.000
gap             0.000     0.000  0.000           5.0             5.000        1.000
multi           0.133     0.667  0.500           5.0             3.667        0.667
semantic        0.100     1.000  0.145           5.0             5.000        0.333


# Этап 6. Оценка качества и диагностика - Report

Проект выполнялся на выделенной ВМ. После превышения лимита вынужден был перейти на локальную разработку (на CPU).
Предлагается увеличить текущий выделенный ресурс как минимум в Pi раз.

Не проверялась подготовка контекста с использованием HyDE - ввиду недостатка вычислительных ресурсов, а также в связи со значительным превышением сроков выполнения проекта.

Результат: сводный профиль качества, диагностика.

Вывод:
Система пока не готова к пилоту. Требуется повысить correctness - до 0.9.
Система хорошо реагитрует на запросы типа exact & gap. Однако, по вопросам типа semantic & multi требуется дополнительной оптимизация.
Рекомендуется начать с повышения MRR за счет реализации "реранкер cross-encoder".

Рекомендации:
1. Реализовать реранкер cross-encoder
2. Протестировать реализацию гибридного поиска с BM25 + слияние результатов через RRF
3. Мигрировать на Qdrant при переходе на промышленную эксплуатацию
4. Планировать время на оптимизацию RAG с запасом 